### Team 41 Members:
1. Joel Arturo Becerril Balderas - $A01797427$
2. Angel Eduardo Urueta Puello - $A01796724$
3. Marco Antonio Chávez García - $A01797547$
4. Efraín Paredes Balgañón - $A01351304$

# TC 5033
## Deep Learning
## Convolutional Neural Networks
#### Activity 2b: Building a CNN for CIFAR10 dataset with PyTorch


# TC 5033
## Deep Learning
## Convolutional Neural Networks
<br>

#### Activity 2b: Building a CNN for CIFAR10 dataset with PyTorch
<br>

- Objective

    The main goal of this activity is to further your understanding of Convolutional Neural Networks (CNNs) by building one using PyTorch. You will apply this architecture to the famous CIFAR10 dataset, taking what you've learned from the guide code that replicated the Fully Connected model in PyTorch (Activity 2a).

- Instructions
    This activity requires submission in teams of 5 or 6 members. Submissions from smaller or larger teams will not be accepted unless prior approval has been granted (only due to exceptional circumstances). While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Understand the Guide Code: Review the guide code from Activity 2a that implemented a Fully Connected model in PyTorch. Note how PyTorch makes it easier to implement neural networks.

    Familiarize Yourself with CNNs: Take some time to understand their architecture and the rationale behind using convolutional layers.

    Prepare the Dataset: Use PyTorch's DataLoader to manage the dataset. Make sure the data is appropriately preprocessed for a CNN.

    Design the CNN Architecture: Create a new architecture that incorporates convolutional layers. Use PyTorch modules like nn.Conv2d, nn.MaxPool2d, and others to build your network.

    Training Loop and Backpropagation: Implement the training loop, leveraging PyTorch’s autograd for backpropagation. Keep track of relevant performance metrics.

    Analyze and Document: Use Markdown cells to explain your architectural decisions, performance results, and any challenges you faced. Compare this model with your previous Fully Connected model in terms of performance and efficiency.

- Evaluation Criteria

    - Understanding of CNN architecture and its application to the CIFAR10 dataset
    - Code Readability and Comments
    - Appropriateness and efficiency of the chosen CNN architecture
    - Correct implementation of Traning Loop and Accuracy Function
    - Model's performance metrics on the CIFAR10 dataset (at least 65% accuracy)
    - Quality of Markdown documentation

- Submission

Submit via Canvas your Jupyter Notebook with the CNN implemented in PyTorch. Your submission should include well-commented code and Markdown cells that provide a comprehensive view of your design decisions, performance metrics, and learnings.

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.utils.data.sampler as sampler
import torchvision.datasets as datasets
import torchvision.transforms as T
import matplotlib.pyplot as plt

# Seleccion de Hardware
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Entrenando en: {device}')

Entrenando en: cpu


### 1. Data Loading & Preprocessing

In [3]:
DATA_PATH = r'./data'
NUM_TRAIN = 50000
NUM_VAL = 5000
NUM_TEST = 5000
MINIBATCH_SIZE = 64

transform_cifar = T.Compose([
    T.ToTensor(),
    T.Normalize([0.491, 0.482, 0.447], [0.247, 0.243, 0.261])
])

# Train dataset
cifar10_train = datasets.CIFAR10(DATA_PATH, train=True, download=True, transform=transform_cifar)
train_loader = DataLoader(cifar10_train, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

# Validation set
cifar10_val = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
val_loader = DataLoader(cifar10_val, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_VAL)))

# Test set
cifar10_test = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
test_loader = DataLoader(cifar10_test, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_VAL, len(cifar10_test))))

100.0%
c:\Users\baldj\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


### 2. Evaluation Metric (Accuracy)
This function evaluates the model on the provided dataset. We disable gradients using `torch.no_grad()` to save memory and set the model to `eval()` mode.

In [4]:
def accuracy(model, loader):
    num_correct = 0
    num_total = 0
    model.eval() # Set model to evaluation mode
    model = model.to(device=device)
    
    with torch.no_grad(): # Disable gradients for evaluation
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            scores = model(x)
            _, preds = scores.max(dim=1)
            
            num_correct += (preds == y).sum()
            num_total += preds.size(0)
            
        return float(num_correct)/num_total

### 3. Training Loop
We compute the cost using `F.cross_entropy` directly during each iteration, mirroring the fundamental tutorials for clear functional step-by-step optimization.

In [5]:
def train(model, optimiser, epochs=100):
    model = model.to(device=device)
    
    for epoch in range(epochs):
        for i, (x, y) in enumerate(train_loader):
            model.train() # Set to training mode
            
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            scores = model(x)
            cost = F.cross_entropy(input=scores, target=y)
            
            optimiser.zero_grad()
            cost.backward()
            optimiser.step()
            
        acc = accuracy(model, val_loader)
        print(f'Epoch: {epoch+1}, Costo: {cost.item():.4f}, Val Acc: {acc:.4f}')

### 4. Baseline Model (Fully Connected)
Flattening a 3D image destroys spatial context. We build this baseline to demonstrate that it plateaus around 50% accuracy, highlighting the need for Convolutional layers.

In [6]:
input_size = 3 * 32 * 32 
hidden_size = 512
num_classes = 10

model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=input_size, out_features=hidden_size),
    nn.ReLU(),
    nn.Linear(in_features=hidden_size, out_features=num_classes)
)

optimiser1 = torch.optim.Adam(model1.parameters(), lr=1e-3)

print('--- Entrenando Modelo Lineal Base ---')
train(model1, optimiser1, epochs=10)

--- Entrenando Modelo Lineal Base ---
Epoch: 1, Costo: 1.5942, Val Acc: 0.4660
Epoch: 2, Costo: 1.0642, Val Acc: 0.4664
Epoch: 3, Costo: 1.1455, Val Acc: 0.4878
Epoch: 4, Costo: 1.9139, Val Acc: 0.4942
Epoch: 5, Costo: 1.4860, Val Acc: 0.4958
Epoch: 6, Costo: 0.8863, Val Acc: 0.4954
Epoch: 7, Costo: 0.6619, Val Acc: 0.5090
Epoch: 8, Costo: 0.8022, Val Acc: 0.4792
Epoch: 9, Costo: 1.2492, Val Acc: 0.5068
Epoch: 10, Costo: 1.0434, Val Acc: 0.5136


### 5. Convolutional Neural Network (CNN)
We construct modular convolutional blocks (`Conv2d -> BatchNorm -> ReLU -> MaxPool`) and chain them in an `nn.Sequential` container, completely aligned with the class examples.

In [7]:
def crear_bloque_conv(in_channels, out_channels, kernel_size=3, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

canales_finales = 64
alto_final = 8
ancho_final = 8
neuronas_planas = canales_finales * alto_final * ancho_final

modelCNN1 = nn.Sequential(
    crear_bloque_conv(in_channels=3, out_channels=32),
    crear_bloque_conv(in_channels=32, out_channels=64),
    
    nn.Flatten(),
    nn.Linear(in_features=neuronas_planas, out_features=256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(in_features=256, out_features=10)
)

optimiserCNN = torch.optim.Adam(modelCNN1.parameters(), lr=1e-3, weight_decay=1e-4)

print('\n--- Entrenando Modelo Convolucional (CNN) ---')
train(modelCNN1, optimiserCNN, epochs=10)


--- Entrenando Modelo Convolucional (CNN) ---
Epoch: 1, Costo: 1.1778, Val Acc: 0.5854
Epoch: 2, Costo: 1.5585, Val Acc: 0.6500
Epoch: 3, Costo: 1.2629, Val Acc: 0.6694
Epoch: 4, Costo: 0.8779, Val Acc: 0.6990
Epoch: 5, Costo: 0.6759, Val Acc: 0.6996
Epoch: 6, Costo: 0.8406, Val Acc: 0.7128
Epoch: 7, Costo: 0.6734, Val Acc: 0.7280
Epoch: 8, Costo: 1.2066, Val Acc: 0.7246
Epoch: 9, Costo: 0.4491, Val Acc: 0.7288
Epoch: 10, Costo: 1.1247, Val Acc: 0.7470


### 6. Final Evaluation
Testing the ultimate accuracy on the unseen Test Set.

In [8]:
test_acc = accuracy(modelCNN1, test_loader)
print(f'\n🏆 Precisión Final en Datos de Prueba (Test Set): {test_acc:.2%}')


🏆 Precisión Final en Datos de Prueba (Test Set): 74.78%
